# 05 — Kiểm tra artifact và kích hoạt model

Notebook này nạp candidate đã train, thử đường suy luận và kiểm tra đủ báo cáo trước khi ghi `Models/current_model.json`. Việc kích hoạt mặc định tắt. Chạy `04_evaluate.ipynb` và `06_test_model.ipynb` trước khi bật cờ ở cuối.


In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa từ 03 mà không chạy train.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

import json
from datetime import datetime


Configured for 88-core CPU
Gốc dự án: /home/ubuntu/sepcung/02.SDC
Đã nạp hàm SDC từ 03_train_model.ipynb


## 1. Chọn và kiểm tra candidate


In [3]:
MODEL_RUN = "20260915_132703_verified_tiered"       # ví dụ: "20260913_162157_verified_tiered"
ACTIVATE_CANDIDATE = False
RUN_DEMO = False

run_dir = select_run_dir(MODEL_RUN or globals().get("run_dir"))
required = ["model.joblib", "meta.json", "thresholds.csv", "oof_pred.parquet"]
missing = [name for name in required if not (run_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Candidate {run_dir} thiếu artifact: {missing}")
predictor = Predictor(run_dir, enrolled={})
print("Candidate:", predictor.run_dir)
print("Contract:", predictor.contract_format)
print("Heads:", predictor.heads)
print("Catalog:", predictor.semantic_type.catalog_version if predictor.semantic_type else None)
display(pd.read_csv(run_dir / "thresholds.csv"))


Candidate: /home/ubuntu/sepcung/02.SDC/Models/20260915_132703_verified_tiered
Contract: sdc-tiered-v2
Heads: ['make', 'type', 'model']
Catalog: 2026-09-12.1


,threshold,id_coverage,id_accuracy_answered,ood_abstain,n_id,n_ood,target_accuracy,target_reached,objective,overridden,head,n_sources
0,1.00,0.987642,1.0,0.925561,2023,2230,0.99,True,0.913203,False,make,1
1,0.96,0.985370,1.0,0.999772,2529,4392,0.99,True,0.985142,False,make,2
2,0.92,0.963883,1.0,0.995839,886,1442,0.99,True,0.959722,False,make,3
3,0.48,0.968750,1.0,0.976000,96,125,0.99,True,0.944750,False,make,4
4,1.00,0.985707,1.0,1.000000,2029,2229,0.90,True,0.985707,False,type,1
5,1.00,0.960652,1.0,1.000000,2516,4392,0.90,True,0.960652,False,type,2
6,0.72,0.986456,1.0,0.947989,886,1442,0.90,True,0.934445,False,type,3
7,0.80,0.791667,1.0,1.000000,96,125,0.90,True,0.791667,True,type,4
8,1.00,0.985171,1.0,0.925561,2023,2230,0.99,True,0.910731,False,model,1
9,1.00,0.959272,1.0,1.000000,2529,4392,0.99,True,0.959272,False,model,2


## 2. Thử suy luận

Ví dụ dùng `Predictor` cho một cửa sổ, `DeviceTracker` cho nhiều cửa sổ và thu nạp tạm không ghi đĩa. Bật `RUN_DEMO` nếu muốn chạy ví dụ đầy đủ.


In [4]:
def demo(run=None):
    """Thử một cửa sổ, DeviceTracker và thu nạp tạm bằng candidate đã chọn."""
    chosen = select_run_dir(run)
    predictor = Predictor(chosen, enrolled={})
    print(f"Dùng run {chosen.name}   ngưỡng {predictor.thresholds}\n")

    echo = [
        {"proto": "dhcp", "opt55": "1,33,3,6,15,28,51,58,59,119",
         "opt60": "dhcpcd-6.8.2:Linux-4.4.22+:armv7l:MT8167B"},
        {"proto": "dns", "qname": "dss-na.amazon.com"},
        {"proto": "dns", "qname": "device-metrics-us.amazon.com"},
        {"proto": "tls", "version": "0303", "ciphers": "49196,49200,159,52393,52392",
         "alpn": "http/1.1", "sni": "device-metrics-us.amazon.com"},
    ]
    for head, r in predictor.predict(echo).items():
        if head not in predictor.heads:
            continue
        print(f"  {head:7s} {r['status']:7s} fp={r['fp']:9s} {str(r['top1']):28s} "
              f"{r['confidence']:.3f}  via {r['retrieval_mode']}")

    def show(title, windows):
        tracker = DeviceTracker(predictor)
        for w in windows:
            tracker.observe(w)
        st = tracker.status()
        print(f"\n{title}")
        print(f"  {st['windows']} cửa sổ, vân tay thu nạp được: {st['enroll_mode']}"
              f"{' (tạm thời)' if st['enroll_provisional'] else ''}")
        for head in predictor.heads:
            s = st[head]
            print(f"  {head:7s} {s['state']:22s} top1={str(s['top1']):16s} "
                  f"cần xử lý={str(s['needs_attention']):5s} thu nạp={s['should_enroll']}")
            if s["remedy"]:
                print(f"          -> {s['remedy']}")
        return tracker

    # MAC lạ hoàn toàn nhưng chỉ nói DNS: không có vân tay nào để thu nạp
    show("MAC lạ, chỉ có DNS:",
         [[{"proto": "dns", "qname": "mqtt.some-new-vendor.io"}]] * 3)

    # MAC lạ có TLS: thu nạp được, nhưng chỉ ở mức tạm thời vì thiếu DHCP
    new_dev = [{"proto": "dns", "qname": "api.some-new-vendor.io"},
               {"proto": "tls", "version": "0303", "ciphers": "111,222,333",
                "alpn": "h2", "sni": "api.some-new-vendor.io"}]
    tracker = show("MAC lạ, có TLS:", [new_dev] * 3)

    print("\n  Người vận hành xác nhận rồi thu nạp:")
    for head, r in tracker.enroll({"make": "SomeNewVendor"}, persist=False).items():
        print(f"    {head}: {r}")
    after = tracker.p.predict(new_dev)["make"]
    print(f"    tra lại -> {after['status']} {after['top1']} via {after['retrieval_mode']}"
          f" ({after['source']})")

In [5]:
sample_records = [{"proto": "mdns", "qname": "Office-Printer._ipp._tcp.local"}]
sample_result = predictor.predict(sample_records)
display(pd.DataFrame({head: {
    "status": sample_result[head]["status"],
    "top1": sample_result[head]["top1"],
    "source": sample_result[head].get("source"),
    "confidence": sample_result[head]["confidence"],
    "reason": sample_result[head]["decision_reason"],
} for head in predictor.heads}).T)
if RUN_DEMO:
    demo(run_dir)


,status,top1,source,confidence,reason
make,abstain,None,model,0.64,insufficient_sources
type,answer,Printer,semantic,0.5714,semantic_type_evidence
model,abstain,None,model,0.72,insufficient_sources


## 3. Kích hoạt sau kiểm thử

Cell này chỉ ghi `Models/current_model.json` khi `ACTIVATE_CANDIDATE=True`, và chỉ sau khi có báo cáo nội bộ cùng hai bài đo từ `06`. Nếu vừa chạy `06`, quay lại chạy lại cell này.


In [6]:
def validate_candidate_reports(run_dir):
    """Kiểm tra read-only ba báo cáo của đúng candidate trước khi kích hoạt."""
    internal_path = REPORTS / f"open_set_{run_dir.name}" / "summary.json"
    external_dir = ROOT / "test_model" / "data_test" / f"out_nb_{run_dir.name}"
    sentinel_path = external_dir / "summary.json"
    cic_path = external_dir / "cic_summary.json"
    for path in (internal_path, sentinel_path, cic_path):
        if not path.is_file():
            raise FileNotFoundError(f"Thiếu báo cáo {path}; chạy 04 và 06 trước")
    internal = json.loads(internal_path.read_text(encoding="utf-8"))
    sentinel = json.loads(sentinel_path.read_text(encoding="utf-8"))
    cic = json.loads(cic_path.read_text(encoding="utf-8"))
    if internal.get("run_id") != run_dir.name or internal.get("external_test_used_for_tuning") is not False:
        raise RuntimeError("Báo cáo nội bộ không khớp candidate")
    if Path(sentinel.get("run", "")).resolve() != run_dir.resolve() or cic.get("run_id") != run_dir.name:
        raise RuntimeError("Báo cáo external không khớp candidate")
    if internal.get("n_sessions", 0) < 1 or sentinel.get("n_device", 0) < 1 or cic.get("n_devices", 0) < 1:
        raise RuntimeError("Báo cáo rỗng")
    if set(sentinel.get("heads", {})) != set(predictor.heads) or set(cic.get("heads", {})) != set(predictor.heads):
        raise RuntimeError("Báo cáo thiếu head make/type/model")
    if sentinel.get("policy_violation") != 0:
        raise RuntimeError("IoT Sentinel có policy violation")
    silent_errors = {head: sentinel["heads"][head].get("silent_error") for head in predictor.heads}
    if any(value is None or value > 0 for value in silent_errors.values()):
        raise RuntimeError(f"IoT Sentinel chưa chứng minh được lỗi im lặng bằng 0: {silent_errors}")
    if cic.get("wrong_count") != 0:
        raise RuntimeError(f"CIC còn {cic['wrong_count']} nhãn sai")
    return {"internal": internal, "sentinel": sentinel, "cic": cic}


try:
    report_check = validate_candidate_reports(run_dir)
except (FileNotFoundError, RuntimeError, ValueError) as exc:
    report_check = None
    if ACTIVATE_CANDIDATE:
        raise
    print("Chưa thể kích hoạt:", exc)
else:
    print("Đủ báo cáo cho candidate:", run_dir.name)

if ACTIVATE_CANDIDATE:
    if run_dir.parent.resolve() != MODELS.resolve():
        raise ValueError("Chỉ kích hoạt model trong Models/<run_id>")
    manifest = {"format": MANIFEST_FORMAT, "run": run_dir.name,
                "updated": datetime.now().isoformat(timespec="seconds")}
    MODELS.mkdir(parents=True, exist_ok=True)
    CURRENT_MODEL_MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Đã kích hoạt:", Predictor().run_dir)
else:
    print("Candidate chưa kích hoạt:", run_dir)


Chưa thể kích hoạt: Thiếu báo cáo /home/ubuntu/sepcung/02.SDC/test_model/data_test/out_nb_20260915_132703_verified_tiered/summary.json; chạy 04 và 06 trước
Candidate chưa kích hoạt: /home/ubuntu/sepcung/02.SDC/Models/20260915_132703_verified_tiered
